# Part 4 - Time-resolved heterogeneous queries

Each query joins several layers of the single graph. Because the second aspect is
now *time*, the queries gain a temporal axis for free: the same join, evaluated
layer by layer, shows *when* a pattern appears. Against separate signaling,
regulatory, and complex tables each of these would need manual per-timepoint
bookkeeping; here each is a few lines.

In [1]:
import pandas as pd

import uc2b_common as uc

G = uc.load()
E = G.views.edges().to_pandas()
E = E[E.ml_kind == "intra"].copy()


def time_of(repr_str):
    inner = repr_str[repr_str.find("(", 2):]
    for t in uc.ALL_TIMES:
        if f"'{t}'" in inner:
            return t
    return None

E["t"] = E["source"].map(time_of)
E["src"] = E["source"].map(uc.bare_vid)
E["tgt"] = E["target"].map(uc.bare_vid)
print("intra-layer edges:", len(E), "| by mechanism:\n", E.edge_kind.value_counts().to_string())

intra-layer edges: 52909 | by mechanism:
 edge_kind
signaling     26330
regulatory    10162
complex        3446


## Q1 - When do transcription factors co-regulate complex assembly?

This join spans three layers: complex hyperedges (protein in complex, baseline),
translation coupling (gene to protein), and regulatory edges (TF to target).
Evaluated per timepoint, it keeps each TF whose *responsive-at-t* targets cover at
least half of a complex's subunits. Running it across the timepoints shows the
co-regulation program expand as the response unfolds - the same query UC2 could
only evaluate statically.

In [2]:
subunits = uc.complex_subunits(G, min_size=3)
reg = E[E.edge_kind == "regulatory"]

def coreg_pairs(t):
    r = reg[reg.t == t]
    tf_targets = (r.assign(tf=r.src.str.removeprefix("gene:"), g=r.tgt.str.removeprefix("gene:"))
                  .groupby("tf")["g"].apply(set))
    return {(tf, cid) for cid, genes in subunits.items() for tf, tgts in tf_targets.items()
            if len(genes & tgts) / len(genes) >= 0.5 and len(genes & tgts) >= 2}

growth = pd.DataFrame([{"time": t, "coreg_pairs": len(coreg_pairs(t))} for t in uc.TIMES])
print("co-regulation (TF, complex) pairs over time:")
print(growth.to_string(index=False))

# Detail at the latest timepoint.
late = coreg_pairs("96h")
detail = pd.DataFrame(sorted(late), columns=["tf", "complex_id"])
detail.to_csv(uc.TABLES / "coreg_pairs_96h.csv", index=False)
print(f"\n96h pairs: {len(late):,} | e.g.")
print(detail.head(8).to_string(index=False))

co-regulation (TF, complex) pairs over time:
time  coreg_pairs
  1h            3
 12h           53
 24h           63
 48h          130
 72h          132
 96h          278

96h pairs: 278 | e.g.
  tf                                                              complex_id
CUX1                                          cpx:ECT2-KIF23-RACGAP1 complex
E2F1 cpx:BASC (Ab 81) complex (BRCA1-associated genome surveillance complex)
E2F1                                                     cpx:BRCA1 A complex
E2F1                                                     cpx:BRCA1 C complex
E2F1                                               cpx:BVR-PKCD-ERK2 complex
E2F1                                             cpx:CDC2-CCNA2-CDK2 complex
E2F1                                       cpx:CDKN1A-TP53-CDK1-PCNA complex
E2F1                                      cpx:CyclinD3-CDK4-CDK6-p21 complex


## Q2 - Cross-compartment signaling, and Q3 - hub complexes

Q2 keeps signaling edges whose endpoints sit in different organelle compartments,
per timepoint, showing cross-compartment traffic rising with the response. Q3 ranks
baseline complexes by the summed signaling degree of their members across all
timepoints.

In [3]:
V = G.views.vertices().to_pandas()
comp = dict(zip(V.vertex_id, V.compartment))
sig = E[E.edge_kind == "signaling"].assign(sc=lambda d: d.src.map(comp), tc=lambda d: d.tgt.map(comp))
cross = sig[(sig.sc != sig.tc) & sig.sc.notna() & sig.tc.notna()]
q2 = (cross.groupby("t").size().reindex(uc.TIMES).fillna(0).astype(int).rename("cross_compartment")
      .to_frame().join(sig.groupby("t").size().reindex(uc.TIMES).fillna(0).astype(int).rename("signaling")))
q2["frac"] = (q2.cross_compartment / q2.signaling.clip(lower=1)).round(3)
print("cross-compartment signaling per time:")
print(q2.to_string())

deg = pd.concat([sig.src.value_counts(), sig.tgt.value_counts()], axis=1).fillna(0).sum(axis=1)
hubs = pd.DataFrame([
    {"complex_id": cid, "n_members": len(m),
     "agg_signaling_degree": float(sum(deg.get(f"prot:{g}", 0) for g in m))}
    for cid, m in uc.complex_subunits(G).items()]).sort_values("agg_signaling_degree", ascending=False)
hubs.head(10).to_csv(uc.TABLES / "hub_complexes.csv", index=False)
print("\ntop hub complexes:\n", hubs.head(5).to_string(index=False))

cross-compartment signaling per time:
     cross_compartment  signaling   frac
t                                       
1h                  85        137  0.620
12h               1183       1732  0.683
24h               1792       2668  0.672
48h               3162       4885  0.647
72h               3816       5988  0.637
96h               6565      10920  0.601



top hub complexes:
                                                                                               complex_id  n_members  agg_signaling_degree
cpx:Polycystin-1 multiprotein complex (ACTN1, CDH1, SRC, JUP, VCL, CTNNB1, PXN, BCAR1, PKD1, PTK2, TLN1)         11                1799.0
                                                                             cpx:CDC2-CCNA2-CDK2 complex          3                1628.0
                                                                       cpx:CDKN1A-TP53-CDK1-PCNA complex          3                1486.0
                                                                              cpx:CDCP1-Src-EGFR complex          3                1434.0
                                                                                 cpx:CAS-SRC-FAK complex          3                1358.0
